# Baseline: CQT + Global Average + Cosine Similarity

Classical MIR baseline for music retrieval. Uses Constant-Q Transform (CQT) with temporal averaging and cosine similarity.

**Limitations:** Fails under tempo changes, key shifts, and short queries.

In [ ]:
import librosa
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

PROCESSED_DIR = Path("data/processed")

def extract_cqt_embedding(path):
    """Extract CQT-based embedding with global average pooling."""
    y, sr = librosa.load(path, sr=22050)
    cqt = librosa.cqt(y, sr=sr)
    cqt = np.abs(cqt)
    embedding = np.mean(cqt, axis=1)
    return embedding

In [ ]:
# Build database of CQT embeddings
database = {}
for f in sorted(PROCESSED_DIR.glob("*.wav")):
    emb = extract_cqt_embedding(str(f))
    database[f.name] = emb

print(f"Indexed {len(database)} pieces")

In [ ]:
# Query: use a snippet as query
query_path = "data/processed/piece_001.wav"  # or your query file
query_emb = extract_cqt_embedding(query_path)

scores = []
for name, emb in database.items():
    sim = cosine_similarity(
        query_emb.reshape(1, -1),
        emb.reshape(1, -1)
    )[0][0]
    scores.append((name, sim))

scores.sort(key=lambda x: x[1], reverse=True)
print("Top 5 matches:")
for name, sim in scores[:5]:
    print(f"  {name}: {sim:.4f}")